# AGAR-RL: Autonomous Multi-Agent Deep Reinforcement Learning Pipeline

Pipeline d'entraînement et d'évaluation Deep Reinforcement Learning (PPO & Prioritized Fictitious Self-Play) sur **Google Colab** (GPU T4/L4/A100) avec **sauvegarde Google Drive** et **inspection des modèles**.

## 0. Connexion & Test de Sauvegarde sur Google Drive
Montez votre Google Drive. Tous les modèles entraînés y sont synchronisés en temps réel.

In [ ]:
# 1. Montage de Google Drive
from google.colab import drive
import os, time

drive.mount('/content/drive')

# 2. Dossier de sauvegarde dédié
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# 3. Test de lecture/écriture
test_file = os.path.join(DRIVE_BACKUP_DIR, 'test_connection.txt')
with open(test_file, 'w', encoding='utf-8') as f:
    f.write(f'Connexion Google Drive OK - {time.ctime()}\n')

print(f'✅ Google Drive connecté avec succès.')
print(f'📁 Dossier de sauvegarde : {DRIVE_BACKUP_DIR}')


## 0-B. Inspection des Modèles Sauvegardés sur Google Drive
Cette cellule scanne l'ensemble de votre dossier Drive et affiche la liste ordonnée des checkpoints, leurs tailles et le dernier modèle final disponible.

In [ ]:
import os, glob, re, time

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup'
zip_files = glob.glob(os.path.join(DRIVE_BACKUP_DIR, '*.zip'))

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

zip_files.sort(key=extract_step)

print('=' * 75)
print(f'📂 CONTENU DU GOOGLE DRIVE : {DRIVE_BACKUP_DIR}')
print('=' * 75)

if not zip_files:
    print('⚠️ Aucun checkpoint (.zip) trouvé pour le moment sur Google Drive.')
else:
    print(f'Total de checkpoints trouvés : {len(zip_files)}\n')
    for z in zip_files:
        sz_mb = os.path.getsize(z) / (1024 * 1024)
        mtime = time.ctime(os.path.getmtime(z))
        step_num = extract_step(z)
        tag = '⭐ [MODÈLE FINAL 5M]' if step_num >= 5000000 else f'[Step {step_num:,}]'
        print(f'{tag:<25} | {os.path.basename(z):<30} | {sz_mb:.1f} Mo | {mtime}')
    
    latest_model = zip_files[-1]
    print('=' * 75)
    print(f'🎯 DERNIER MODÈLE LE PLUS AVANCÉ : {latest_model}')
    print('=' * 75)


## 1. Détection de l'Environnement et Synchronisation du Code

In [ ]:
import os, sys

# 1. Récupération des dernières modifications ou clonage
if os.path.exists('.git'):
    print('🔄 Récupération des dernières mises à jour du repo...')
    !git pull origin main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et mise à jour...')
    %cd agario
    !git pull origin main
else:
    print('🌐 Environnement distant Colab détecté. Clonage du repo...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Configuration du PYTHONPATH et installation des dépendances Farama Gymnasium
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt tensorboard
!apt-get install -qq -y ffmpeg

# 3. Vérification GPU CUDA
import torch
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'🚀 GPU actif : {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ ATTENTION : Vous êtes sur CPU ! Cliquez sur Exécution > Modifier le type d\'exécution > GPU T4/L4.')


## 2. Validation de la Suite de Tests (27 Tests)

In [ ]:
!python -m pytest -v

## 3. Évaluation & Replay Vidéo HD du VRAI Modèle Final Entraîné (5M Steps)
Cette cellule charge spécifiquement le modèle le plus entraîné (`ppo_step_5000000.zip` ou `ppo_final.zip`), enregistre un match HD de 2400 steps (80 secondes) et affiche la vidéo ainsi que les statistiques complètes de jeu.

In [ ]:
import os, glob, re
from IPython.display import HTML, display
from base64 import b64encode

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

# 1. Recherche ciblée du modèle final (priorité aux 5M de steps)
candidates = [
    '/content/drive/MyDrive/agario_rl_backup/ppo_step_5000000.zip',
    'checkpoints/self_play_pool/ppo_step_5000000.zip',
    '/content/drive/MyDrive/agario_rl_backup/ppo_final.zip',
    'checkpoints/ppo/ppo_final.zip',
]
target_model = next((c for c in candidates if os.path.exists(c)), None)

if not target_model:
    all_ckpts = glob.glob('/content/drive/MyDrive/agario_rl_backup/*.zip') + glob.glob('checkpoints/self_play_pool/*.zip')
    if all_ckpts:
        all_ckpts.sort(key=extract_step)
        target_model = all_ckpts[-1]
    else:
        target_model = 'checkpoints/ppo/ppo_latest.zip'

step_count = extract_step(target_model)
print('=' * 70)
print(f'🎬 MODÈLE SÉLECTIONNÉ : {target_model}')
print(f'📊 Palier de pas : {step_count:,} steps')
print('=' * 70)

# 2. Enregistrement du Replay HD (2400 frames @ 30 FPS = 80 secondes)
os.makedirs('recordings', exist_ok=True)
!python src/inference/record_match.py \
    --model "{target_model}" \
    --output recordings/eval_match_final.mp4 \
    --steps 2400

# 3. Sauvegarde sur Google Drive
if os.path.exists('recordings/eval_match_final.mp4') and os.path.exists('/content/drive/MyDrive/agario_rl_backup'):
    !cp recordings/eval_match_final.mp4 /content/drive/MyDrive/agario_rl_backup/eval_match_final.mp4
    print('📁 Vidéo copiée sur Google Drive dans : agario_rl_backup/eval_match_final.mp4')

# 4. Affichage direct dans le Notebook
video_path = 'recordings/eval_match_final.mp4'
if os.path.exists(video_path):
    mp4_bytes = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="800" height="450" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    print(f'Taille de la vidéo : {os.path.getsize(video_path) / 1_000_000:.1f} Mo')
else:
    print('⚠️ Erreur : vidéo non générée.')


## 4. Choix Stratégique : Continuer l'Entraînement ou Repartir de Zéro ?

Après avoir visionné la vidéo du modèle 5M ci-dessus, deux approches sont possibles :

### Option A : Poursuivre le Fine-Tuning à partir du modèle 5M (Recommandé si la navigation de base est solide)
Le modèle conserve ses acquis de collecte et de survie, et intègre les nouveaux signaux (anti-suicide split, lissage 180°, gradient de danger dominant).

### Option B : Repartir de zéro avec les règles V2 (Recommandé si le modèle a trop d'habitudes indésirables)
Le modèle apprend dès le premier pas avec les contraintes saines (interdiction de split sous 36 de masse, pas d'oscillations, évitement strict des murs et prédateurs).

In [ ]:
# ==========================================================================
# OPTION A : CONTINUER L'ENTRAÎNEMENT DU MODÈLE 5M (FINE-TUNING V2)
# Décommentez et lancez cette cellule pour continuer à partir du modèle 5M
# ==========================================================================

!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 8000000 \
    --pool-interval 200000 \
    --backup-dir /content/drive/MyDrive/agario_rl_backup \
    --resume /content/drive/MyDrive/agario_rl_backup/ppo_step_5000000.zip \
    --device auto

# ==========================================================================
# OPTION B : NOUVEL ENTRAÎNEMENT COMPLET DEPUIS ZÉRO (V2 FRESH START)
# Pour repartir de zéro, remplacez --resume par 'none' :
# !python src/training/train_colab.py \
#     --n-envs 16 \
#     --total-timesteps 8000000 \
#     --pool-interval 200000 \
#     --backup-dir /content/drive/MyDrive/agario_rl_backup_v2 \
#     --resume none \
#     --device auto
# ==========================================================================


## 5. Exporter la Politique vers ONNX (< 0.02 ms de latence CPU)
Exporte le meilleur modèle PyTorch au format ONNX universel pour déploiement en direct.

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = [
    '/content/drive/MyDrive/agario_rl_backup/ppo_step_5000000.zip',
    'checkpoints/self_play_pool/ppo_step_5000000.zip',
    '/content/drive/MyDrive/agario_rl_backup/ppo_final.zip',
    'checkpoints/ppo/ppo_final.zip',
]
target_model = next((c for c in candidates if os.path.exists(c)), None)
if not target_model:
    all_ckpts = glob.glob('/content/drive/MyDrive/agario_rl_backup/*.zip') + glob.glob('checkpoints/self_play_pool/*.zip')
    all_ckpts.sort(key=extract_step)
    target_model = all_ckpts[-1] if all_ckpts else 'checkpoints/ppo/ppo_latest.zip'

os.makedirs('models', exist_ok=True)
!python src/inference/export_onnx.py \
    --model "{target_model}" \
    --output models/model.onnx

if os.path.exists('models/model.onnx') and os.path.exists('/content/drive/MyDrive/agario_rl_backup'):
    !cp models/model.onnx /content/drive/MyDrive/agario_rl_backup/model.onnx
    print('📁 Modèle ONNX sauvegardé sur Google Drive dans : agario_rl_backup/model.onnx')
